In [22]:
#!pip install sympy

In [23]:
from pyomo.environ import *
import numpy as np
from math import pi
from Sympy_QrOPF_ALM_class import SympyACOPFModel
from Sympy_QrOPF_ALM_class import PrintQHDACOPFResults
from Sympy_QrOPF_ALM_class import solve_with_gurobi_from_sympy
from Sympy_QrOPF_ALM_class import initialize_qhd_acopf_log
from Sympy_QrOPF_ALM_class import append_qhd_acopf_results
from qhdopt import QHD
import json
import io
import contextlib

In [24]:
def load_matpower_json(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    Sbase = float(data["Sbase"])

    # 把 "k1", "k2", ... 转成 int key: 1, 2, ...
    buses = {int(k.replace("k", "")): v for k, v in data["buses"].items()}
    lines = {int(k.replace("k", "")): v for k, v in data["lines"].items()}
    gens  = {int(k.replace("k", "")): v for k, v in data["gens"].items()}

    return Sbase, buses, lines, gens

In [25]:
# 初始化（用默认 3-bus 数据）
#model = SympyACOPFModel()   # 打开 proximal 选项，后面会用到
n = 3  # 选择测试系统的规模，2 或 3 或者更多

if n == 2:
    # 2bus model
    Sbase = 10.0
    buses = {
        1: [1, 0, 1.00, 0.0, 0.0, 0.0, 0.0, 0.0],
        2: [2, 1, 1.01, 0.0, 0.0, 0.0, 0.3, 0.1],
    }
    lines = {
        1: [1, 2, 0.0452, 0.1852, 0.0204, 1.0, 30.0 / Sbase],
    }
    gens = {
        1: [1, 0.0 / Sbase, 20.0 / Sbase, -20.0 / Sbase, 100.0 / Sbase, 0.00375, 2.0, 0.0],
    }
    model = SympyACOPFModel(Sbase = Sbase, buses=buses, lines=lines, gens=gens)

elif n == 3:
    # 3bus model
    model = SympyACOPFModel()
else:
    # n bus model
    Sbase, buses, lines, gens = load_matpower_json(f"case{n}_custom_pretty.json")
    model = SympyACOPFModel(Sbase = Sbase, buses=buses, lines=lines, gens=gens) # 打开 proximal 选项，后面会用到

print("Model initialized with", n, "buses", model.n_lines, "lines and", model.n_gens, "generators.")


Model initialized with 3 buses 3 lines and 2 generators.


In [ ]:
# =========================
#   Linear ALM + QHD Loop
# =========================

import numpy as np

# 1) 构造 h(x)
h_func = model.build_h_func()
model.reset_lambdas(0.0)

# 2) 初始点
xk = model.build_initial_x0()
#xk = np.load("opf_result_ordered.npy", allow_pickle=True)

rho = 5.0
alpha = 5e-3   # 对偶步长尽量小一点
max_outer = 200
tol = 1e-4

# ========= 新增：初始化日志文件 =========
log_file = initialize_qhd_acopf_log(model, folder="logs")
print("Log file:", log_file)

print("\n===== Start Linear ALM Loop =====\n")

for k in range(max_outer):

    print(f"\n--- Outer Iteration {k} ---")

    # ======================================
    # (1) 在线性化点 xk 构造二次 L^(k)(x)
    # ======================================
    Lag, variable_list, Var_bound_list = \
        model.build_linear_ALM_Lagrangian_syms(
            x_center=xk,
            rho=rho,
            ref_bus_id=None,
            mu_prox=10.0
        )
    
    bad_bounds = []
    for i, (var, bnd) in enumerate(zip(variable_list, Var_bound_list)):
        lb, ub = float(bnd[0]), float(bnd[1])
        if ub < lb:
            bad_bounds.append((i, str(var), lb, ub))

    if bad_bounds:
        print("\n=== Invalid bounds found ===")
        for item in bad_bounds:
            print(item)
        raise ValueError("Var_bound_list contains invalid bounds (ub < lb).")

    option = 1  # 1: QHD, 2: Gurobi
    options_str = ["QHD", "Gurobi"]
    print(f"\n--- Outer Iteration {k} ---")

    if option == 1:
        # ======================================
        # (2A) QHD 求解子问题
        # ======================================
        qhd_model = QHD.SymPy(Lag, variable_list, Var_bound_list)


        qhd_solver = "simbi" # openjij / simbi
        
        if option == 1:
            print(f"\n--- you are using QHD with {qhd_solver} solver ---")
        elif option == 2:
            print(f"\n--- you are using Gurobi as solver ---")
        else:
            print(f"\n--- invalid option ---")

        if qhd_solver == "simbi":
            qhd_model.simbi_setup(
                resolution=7,
                agents=2048,
                max_steps=3000,
                embedding_scheme="unary",
                post_processing_method='TNC',
                verbose=True
            )
        else:
            qhd_model.openjij_setup(
                resolution=6,
                shots=1024,
                sampler_name="SQASampler",
                seed=42,
                debug=False,
                sampler_init_kwargs={},
                sample_kwargs={
                    "beta": 5.0,
                    "gamma": 1.0,
                    "trotter": 8,
                    "num_sweeps": 3000,
                    "reinitialize_state": True,
                },
            )
        _qhd_stdout = io.StringIO()
        with contextlib.redirect_stdout(_qhd_stdout):
            response = qhd_model.optimize(refine=True, verbose=0)
        for _line in _qhd_stdout.getvalue().splitlines():
            if _line.startswith("Minimizer:"):
                continue
            if _line.strip().startswith("[") and _line.strip().endswith("]"):
                continue
            if _line.strip():
                print(_line)

        x_new = np.asarray(response.refined_minimizer)

    else:
        # ======================================
        # (2B) 用 Gurobi 求解子问题
        # ======================================
        x_new = solve_with_gurobi_from_sympy(
        L_sym=Lag,
        variable_list=variable_list,
        Var_bound_list=Var_bound_list,
        verbose=False   # 如果想看 Gurobi 日志就改成 True
    )
    PrintQHDACOPFResults(model, x_new)

    # ======================================
    # (3) 计算真实约束
    # ======================================
    h_val = h_func(x_new)
    norm_h = np.linalg.norm(h_val)

    print("||h(x)|| =", norm_h)

    # ======================================
    # (4) 对偶更新
    # ======================================
    lambda_new, h_x = model.update_lambda(
        x_new,
        alpha=alpha,
        h_func=h_func
    )
    # ======================================
    # (5) 自适应 rho
    # ======================================
    h_old = h_func(xk)
    print(f"[rho-check] ||h_old||={np.linalg.norm(h_old):.3e}, ||h_new||={np.linalg.norm(h_val):.3e}, rho={rho:.3g}")
    rho_max = 1024
    if np.linalg.norm(h_x) > 0.5 * np.linalg.norm(h_old) and rho < rho_max:
        rho *= 2
        print("Increasing rho to", rho)

    # ======================================
    # (6) 可行性检查
    # ======================================
    _, check_flag = model.check_constraints(x_new)
    print("Constraint check:", check_flag)
    if check_flag:
        print("\nConverged!")
        break

    # ======================================
    # (7) 记录日志（每轮追加）
    # ======================================
    subs_dict = {var: val for var, val in zip(model.variable_list, x_new)}
    #objective_value = float(model.objective.evalf(subs=subs_dict))

    append_qhd_acopf_results(
        model=model,
        solution=x_new,
        log_file=log_file,
        iteration=k,
        rho=rho,
        alpha=alpha,
        h_x=h_val,
        lambda_vec=lambda_new,
        objective_value=0,
        feasibility=check_flag,
    )

    # 如果你还想同时在屏幕上打印表格，可以再保留这一句
    # PrintQHDACOPFResults(model, x_new, iteration=k, log_file=log_file, print_to_screen=True)


    # ======================================
    # (8) 收敛判据
    # ======================================
    step_norm = np.linalg.norm(x_new - xk)
    if norm_h < tol and step_norm < 1e-5:
        print("\nConverged!")
        break

    # ======================================
    # (9) 更新 primal
    # ======================================
    xk = x_new.copy()

print("\n===== End Loop =====\n")
print("Final log file:", log_file)

Log file: logs\Buses-3-22-50-16-04-06-2026.txt

===== Start Linear ALM Loop =====


--- Outer Iteration 0 ---


Bifurcated agents:   0%|          | 0/2048 [00:03<?, ?it/s]


backend time consumption: 3.671858072280884
 0.85714286 0.85714286 0.85714286 0.85714286 0.71428571 0.85714286
 0.71428571 0.85714286 1.         0.85714286 0.85714286 0.85714286
 0.85714286 0.71428571 0.85714286 0.85714286 0.71428571 0.71428571
 0.71428571 0.71428571 0.71428571 0.85714286 0.85714286 0.85714286
 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286
 0.85714286 0.85714286 0.71428571 0.85714286 0.85714286 0.85714286
 0.85714286 0.85714286 0.85714286 0.71428571 0.85714286 0.85714286
 1.         0.85714286 0.85714286 1.         0.71428571 0.71428571
 0.71428571 0.71428571 0.71428571 0.71428571]
Update time: 2026-04-06 22:52:57
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	0.735015	0.050509	0.977627	0.000000	0.149889	0.000000	0.000000
2	0.579629	0.018614	0.964324	0.000000	0.084310	0.000000	0.000000
3	0.913299	-0.009233	1.035148	0.000000	0.

Bifurcated agents:   0%|          | 0/2048 [00:02<?, ?it/s]


backend time consumption: 2.7643182277679443
 0.71428571 0.85714286 0.71428571 0.71428571 0.85714286 0.71428571
 0.71428571 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286
 0.85714286 0.71428571 0.71428571 0.85714286 0.71428571 0.85714286
 0.85714286 0.71428571 0.85714286 0.85714286 0.85714286 0.85714286
 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286
 0.85714286 0.85714286 0.71428571 0.85714286 0.85714286 0.85714286
 0.71428571 0.71428571 0.85714286 0.71428571 0.85714286 0.85714286
 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286
 0.71428571 0.85714286 0.71428571 0.71428571]
Update time: 2026-04-06 22:55:25
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	0.911511	-0.009023	1.070620	0.000000	0.210643	0.000000	0.000000
2	0.552055	0.174692	1.038828	0.000000	0.273857	0.000000	0.000000
3	0.768682	-0.078823	1.099109	0.000000	

...

backend time consumption: 2.6781845092773438
 0.71428571 0.71428571 0.85714286 0.85714286 0.71428571 0.71428571
 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286
 0.85714286 0.71428571 0.71428571 0.71428571 0.85714286 0.85714286
 0.71428571 0.71428571 0.71428571 0.71428571 0.85714286 0.85714286
 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286 1.
 0.85714286 0.85714286 0.71428571 0.85714286 0.71428571 0.85714286
 0.85714286 0.85714286 0.71428571 0.71428571 0.85714286 0.85714286
 0.85714286 0.85714286 0.85714286 1.         0.85714286 0.71428571
 0.71428571 0.85714286 0.85714286 0.71428571]
Update time: 2026-04-07 06:55:26
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	0.959641	0.423993	1.100000	1.141437	8.935761	0.000000	0.000000
2	0.686011	0.549375	1.100000	1.802756	8.910689	0.000000	0.000000
3	0.795152	-0.696835	1.090973	0.000000	0.000000	

Bifurcated agents:   0%|          | 0/2048 [00:03<?, ?it/s]


backend time consumption: 3.208451271057129
 0.71428571 0.71428571 0.71428571 0.71428571 0.71428571 0.71428571
 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286
 0.85714286 0.71428571 0.85714286 0.85714286 0.85714286 0.71428571
 0.85714286 0.71428571 0.71428571 0.85714286 0.85714286 0.85714286
 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286
 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286
 0.85714286 0.85714286 0.85714286 0.71428571 1.         0.85714286
 0.85714286 0.85714286 0.85714286 0.85714286 0.71428571 0.85714286
 0.85714286 0.85714286 0.71428571 0.71428571]
Update time: 2026-04-07 06:58:29
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	0.895702	0.477457	1.097852	1.684791	9.508860	0.000000	0.000000
2	0.800811	0.508336	1.100000	0.657052	8.744072	0.000000	0.000000
3	0.719197	-0.516681	1.100000	0.000000	0.

Bifurcated agents:   0%|          | 0/2048 [00:02<?, ?it/s]


backend time consumption: 2.7966198921203613
 0.85714286 0.85714286 0.71428571 0.71428571 0.71428571 0.85714286
 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286 1.
 0.85714286 0.71428571 0.85714286 0.71428571 0.85714286 0.71428571
 0.85714286 0.71428571 0.85714286 0.71428571 0.85714286 0.71428571
 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286 0.85714286
 0.85714286 0.85714286 1.         0.85714286 1.         0.85714286
 0.85714286 0.85714286 0.71428571 0.71428571 0.85714286 0.85714286
 0.85714286 0.85714286 0.85714286 1.         0.85714286 0.71428571
 0.71428571 0.85714286 0.71428571 0.71428571]
Update time: 2026-04-07 07:01:13
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	0.961285	0.256399	1.077083	1.985598	7.925087	0.000000	0.000000
2	0.912993	0.316291	1.100000	1.119684	8.360542	0.000000	0.000000
3	0.908391	-0.478235	1.100000	0.000000	0.000000	

In [27]:
Lag, variable_list, Var_bound_list = \
        model.build_linear_ALM_Lagrangian_syms(
            x_center=np.ones_like(xk),
            rho=rho,
            ref_bus_id=None,
            mu_prox=1500.0
        )

In [28]:
Lag

1390.001875*P_G0**2 - 8279.53083820757*P_G0*Wii_II0 - 8279.53083820757*P_G0*Wii_RR0 + 6687.54711025364*P_G0*Wij_II0 + 1591.98372795393*P_G0*Wij_II1 - 20027.8103562284*P_G0*Wij_IR0 - 6522.90677913867*P_G0*Wij_IR1 + 20027.8103562284*P_G0*Wij_RI0 + 6522.90677913867*P_G0*Wij_RI1 + 6687.54711025364*P_G0*Wij_RR0 + 1591.98372795393*P_G0*Wij_RR1 - 1498.00350342549*P_G0 + 1390.00875*P_G1**2 - 8870.62591562839*P_G1*Wii_II1 - 8870.62591562839*P_G1*Wii_RR1 + 6687.54711025364*P_G1*Wij_II2 + 2183.07880537475*P_G1*Wij_II3 - 20027.8103562284*P_G1*Wij_IR2 - 6652.64541216833*P_G1*Wij_IR3 + 20027.8103562284*P_G1*Wij_RI2 + 6652.64541216833*P_G1*Wij_RI3 + 6687.54711025364*P_G1*Wij_RR2 + 2183.07880537475*P_G1*Wij_RR3 - 1498.25628587159*P_G1 + 3950.0*P_ij0**2 + 5120.0*P_ij0*Q_ij0 - 2560.0*P_ij0*S_tot_sq0 - 6687.54711025364*P_ij0*Wii_II0 - 6687.54711025364*P_ij0*Wii_RR0 + 6687.54711025364*P_ij0*Wij_II0 - 20027.8103562284*P_ij0*Wij_IR0 + 20027.8103562284*P_ij0*Wij_RI0 + 6687.54711025364*P_ij0*Wij_RR0 - 6619.05

In [29]:
model.linear_jacobian

Matrix([
[1, 0, 0, 0, -6.46838346734966,                 0,                 0,                 0,                 0,                 0, -6.46838346734966,                 0,                 0, 5.22464617988566, 1.24373728746401,                0,                0,                0,                0,  15.6467268408034,  5.09602092120209,                 0,                 0,                 0,                 0, -15.6467268408034, -5.09602092120209,                 0,                 0,                 0,                 0, 5.22464617988566, 1.24373728746401,                0,                0,                0,                0, 0, 0, 0,        0,        0,        0,        0,        0,        0,        0,        0,        0,        0,        0,        0, 0, 0, 0, 0, 0, 0],
[0, 1, 0, 0,                 0, -6.93017649658468,                 0,                 0,                 0,                 0,                 0, -6.93017649658468,                 0,                0,              

In [30]:
model.G_mat

array([[ 6.46838347, -5.22464618, -1.24373729],
       [-5.22464618,  6.9301765 , -1.70553032],
       [-1.24373729, -1.70553032,  2.9492676 ]])

In [31]:
model.arc_collection

[(0, 1), (1, 0), (0, 2), (2, 0), (1, 2), (2, 1)]

In [32]:
response.refined_minimizer


array([ 1.98559774,  1.11968386,  7.92508721,  8.36054209,  0.92406817,
        0.83355701,  0.82517484,  0.24647226,  0.28877118, -0.43442446,
        0.22214883,  0.62320758,  0.53143678,  0.30960722,  0.41768942,
        0.83687027,  0.62133836,  0.42248146,  0.48840482,  0.67684332,
        0.82505718,  0.70621879,  0.96579177,  0.71203335,  0.96112938,
        0.6707393 ,  0.95005785,  0.78112097,  0.68146449,  1.03140515,
        0.48921248,  0.66662023,  0.29565109,  0.4626437 ,  0.48085448,
        0.58998151,  0.816097  ,  1.16010753,  1.21      ,  1.21      ,
        0.78085242,  2.0086697 ,  1.15637205,  2.04569961, -0.84136613,
       -2.37697915,  2.6558309 ,  2.05037628,  2.05140692,  1.31618484,
        2.35776095,  1.03496631,  7.08260449,  6.82586224,  5.00163391,
        5.91450799,  5.64963927,  6.67231633])

In [33]:
qhd_model.func

645.001875*P_G0**2 - 8279.53083820757*P_G0*Wii_II0 - 8279.53083820757*P_G0*Wii_RR0 + 6687.54711025364*P_G0*Wij_II0 + 1591.98372795393*P_G0*Wij_II1 - 20027.8103562284*P_G0*Wij_IR0 - 6522.90677913867*P_G0*Wij_IR1 + 20027.8103562284*P_G0*Wij_RI0 + 6522.90677913867*P_G0*Wij_RI1 + 6687.54711025364*P_G0*Wij_RR0 + 1591.98372795393*P_G0*Wij_RR1 - 14.8515038177526*P_G0 + 645.00875*P_G1**2 - 8870.62591562839*P_G1*Wii_II1 - 8870.62591562839*P_G1*Wii_RR1 + 6687.54711025364*P_G1*Wij_II2 + 2183.07880537475*P_G1*Wij_II3 - 20027.8103562284*P_G1*Wij_IR2 - 6652.64541216833*P_G1*Wij_IR3 + 20027.8103562284*P_G1*Wij_RI2 + 6652.64541216833*P_G1*Wij_RI3 + 6687.54711025364*P_G1*Wij_RR2 + 2183.07880537475*P_G1*Wij_RR3 - 4.82680250961206*P_G1 + 648.085977729505*P_ij0**2 + 447.309174318796*P_ij0*Q_ij0 - 88.8825235214007*P_ij0*S_tot_sq0 - 6687.54711025364*P_ij0*Wii_II0 - 6687.54711025364*P_ij0*Wii_RR0 + 6687.54711025364*P_ij0*Wij_II0 - 20027.8103562284*P_ij0*Wij_IR0 + 20027.8103562284*P_ij0*Wij_RI0 + 6687.5471102

In [34]:
print(model.P_D, model.Q_D, model.P_G, model.Q_G)
print(model.variable_list)
print(model.Var_bound_list)

[0.  0.  0.3] [0.  0.  0.1] (P_G0, P_G1) (Q_G0, Q_G1)
[P_G0, P_G1, Q_G0, Q_G1, Wii_RR0, Wii_RR1, Wii_RR2, Wii_RI0, Wii_RI1, Wii_RI2, Wii_II0, Wii_II1, Wii_II2, Wij_RR0, Wij_RR1, Wij_RR2, Wij_RR3, Wij_RR4, Wij_RR5, Wij_RI0, Wij_RI1, Wij_RI2, Wij_RI3, Wij_RI4, Wij_RI5, Wij_IR0, Wij_IR1, Wij_IR2, Wij_IR3, Wij_IR4, Wij_IR5, Wij_II0, Wij_II1, Wij_II2, Wij_II3, Wij_II4, Wij_II5, V_sq0, V_sq1, V_sq2, P_ij0, P_ij1, P_ij2, P_ij3, P_ij4, P_ij5, Q_ij0, Q_ij1, Q_ij2, Q_ij3, Q_ij4, Q_ij5, S_tot_sq0, S_tot_sq1, S_tot_sq2, S_tot_sq3, S_tot_sq4, S_tot_sq5]
[[0.0, 2.0], [0.0, 2.0], [-2.0, 10.0], [-2.0, 10.0], [0, 1.21], [0, 1.21], [0, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.21, 1.21], [0, 1.21], [0, 1.21], [0, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.21, 1.21], [-1.